# Proximal operator of the matrix perspective function

$
\newcommand\vp{\varphi}
\newcommand\bR{\mathbb{R}}
\newcommand\bT{\mathbb{T}}
\newcommand\bZ{\mathbb{Z}}
\newcommand\cO{\mathcal{O}}
\newcommand\prox{\mathrm{prox}}
\DeclareMathOperator\diver{div}
\DeclareMathOperator\curl{curl}
\DeclareMathOperator\Id{Id}
\DeclareMathOperator\Tr{Tr}
\DeclareMathOperator\Diag{Diag}
\DeclareMathOperator\sinc{sinc}
\DeclareMathOperator*\argmin{argmin}
\DeclareMathOperator\Fr{Fr}
\newcommand\diff{\mathrm{d}}
\newcommand\<{\langle} \newcommand\>{\rangle}
$

The scalar perspective function is a convex and lower-semi-continuous function defined as  
\begin{align*}
    P : \bR \times \bR^N &\to [0,\infty], &
    (\rho,m)  &\mapsto \frac{|m|^2} {2 \rho} \text{ if }\rho>0,
\end{align*}
with the conventions $P(0,0)=0$ and $P = -\infty$ elsewhere.
In the applications considered in this series of notebooks, this function typically represents the kinetic energy density of some material with with density $\rho$ and momentum density $m$.


The scalar perspective function appears in the energetic formulation gradient-Burgers PDE, but the divergence-Burgers and Euler equations require a variant, referred to as the matrix perspective function, which is defined as 
\begin{align*}
    P : S_d \times (\bR^d)^n & \to [0,\infty], & 
    (\rho,m) & \mapsto \frac 1 2 \Tr(m^\top\rho^{-1} m) \text{ if } \rho \succ 0,
\end{align*}
$P(\rho,m) = \infty$ if $\rho$ has a negative eigenvalue, and the remaining cases correspond to the lower semi-continuous envelope. 

**Proximal operator.**
Recall that the proximal operator is defined as 
$$
    \prox_{\tau f}(x) := \argmin_{y} \frac 1 2 \|x-y\|^2 + \tau f(y),
$$
and amounts to an implicit time step of gradient descent of $f$.

We implemented the proximal operator of the scalar perspective function in the notebook on the [Porous medium equation](PorousMinimization.ipynb)

**Dual problem.**

The following paper studies the dual problem to the proximal operator computation, and is used as foundation for this code. 

* Won and Joong-Ho Johann. *Proximity operator of the matrix perspective function and its applications.* Advances in Neural Information Processing Systems, 33:6305–6314, 2020.

Eventually, one is led to the minimization of the following functional 
$$
    g(\mu) := \Tr(\mu) + \|(X-\mu)_+\|^2_{\Fr},
$$
which is convex, piecewise smooth, has gradient
$$
    \nabla g(\mu) = \Id - (X-\mu)_+
$$
using the Frobenius inner product, and Hessian
$$
    \nabla^2 g(\mu)[\eta,\eta] = \< \Lambda \circ (P^\top \eta P), P^\top \eta P\>,
$$
where $X-\mu = P \Diag(\lambda) P^\top$ is the diagonalization in an orthonormal basis, and 
$$
    \Lambda_{ij} = \frac{\lambda_i^+ + \lambda_j^+}{|\lambda_i| + |\lambda_j|},
$$
with $\Lambda_{ij} := 0$ if $\lambda_i = \lambda_j = 0$.

We use a damped Newton method to solve the dual problem. Note that a diagonalization of the symmetric matrix $X-\mu$ is required at each step, which has substantial computational cost. The paper discusses possible ways to update this eigendecomposition cheaply, but they are not implemented yet.

**Solution reconstruction.**

The connection between the primal and dual problem is as follows.
* Construct the $(n+d)\times (n+d)$ symmetric matrix
$$
X = 
\begin{pmatrix} \Id & m^\top/\sqrt 2 \\
m/\sqrt 2 & -\rho \end{pmatrix}
$$
* Compute the minimizer of $\mu\in S_n \mapsto g(E\mu E^\top)$, where $E$ inserts the values of $\mu$ in the top-left block, and obtain the proximal value $(\rho_*,m_*)$ (assuming $\tau=1$) from the identity
$$
(X-E\mu E^\top)_- = 
\begin{pmatrix} \mu & -m_*^\top/\sqrt 2 \\
-m_*/\sqrt 2 & \rho_* \end{pmatrix}
$$

Denoting by $P_\tau$ the above proximal operator, we have in addition $P_\tau(\rho,m) = \tau P_1(\rho/\tau,m/\tau)$. (This follows from the Moreau Formula, and from the fact that the dual operator is a projection onto a convex set.)

**Rotational invariance**

The perspective function satisfies
$$
    P(\rho,m R)=P(\rho,m)
$$
for any orthogonal matrix $R$.
If $m$ has shape $(d,n)$ with $n>d$, we can use the QR decomposition to write it under the form
$$
    m = m_0 R
$$
where $m_0$ is a $d\times d$ matrix, and $R$ is an orthogonal $d\times n$ matrix.
We then have
$$
    \prox_P(\rho,m) = (\rho',m' R),
$$
where $(\rho',m') = \prox_P(\rho,m_0)$.
Note that the computation of $\prox_P(\rho,m_0)$ involves the diagonalization of $2d\times 2d$ symmetric matrices, as opposed to $(d+n)\times (d+n)$ symmetric matrices in the case of $\prox_P(\rho,m)$.

**Linear algebra routines**
The following linear algebra routines are not available in taichi (v1.7.3), hence had to be reimplemented. The proposed implementation may not be state of the art.
- sym_eig4 : diagonalization of 4x4 symmetric matrices. Warning : proposed implem is long to compile.
- qr : QR decomposition. Warning : proposed implem will fail if the matrix is not full rank.

## 0. Importing the required libraries

In [1]:
import sys
sys.path.insert(0,"/Users/jean-mariemirebeau/Dropbox/Programmes/GithubM1/AGDT/AdaptiveGridDiscretizations_Taichi")
sys.path.insert(0,"/Users/jean-mariemirebeau/Dropbox/Programmes/GithubM1/AdaptiveGridDiscretizations")


import numpy as np
import taichi as ti
ti.init(arch=ti.cpu,default_fp=ti.f64)
from agdt.Proximal import MatrixPerspective as MP
from agdt import Linalg

from agd import AutomaticDifferentiation as ad
from agd import LinearParallel as lp
np.set_printoptions(linewidth=2000)

[Taichi] version 1.7.3, llvm 15.0.7, commit 5ec301be, osx, python 3.11.0
[Taichi] Starting on arch=arm64


[I 04/10/25 17:53:34.804 244517] [shell.py:_shell_pop_print@23] Graphical python shell detected, using wrapped sys.stdout


### 0.1 Python implementation

This implementation was validated independently in another notebook, and is used for reference.

In [2]:
def pyRandomSym(ndim,relax=0.1,shape=tuple()):
    """Generate random symmetric matrices"""
    A = 2*np.random.rand(*shape,ndim,ndim)-1
    M = np.swapaxes(A,-1,-2) @ A
    trM = sum(M[...,i,i] for i in range(ndim))
    M += relax*trM[...,None,None]*np.eye(ndim)
    return M

def _sym_iso(i,j): 
    """Factors used for isometry between Frobenius norm and Euclidean norm."""
    return np.sqrt(2) if i!=j else 1

def fltsym_iso(m):
    """Turns a symmetric matrix into a vector, isometrically w.r.t
    the Frobenius norm and the Euclidean norm."""
    return ad.array([m[i,j]*_sym_iso(i,j) for i in range(len(m)) for j in range(i+1)])

def expsym_iso(v):
    """Turns a vector into a symmetric matrix, isometrically w.r.t 
    the Frobenius norm and the Euclidean norm."""
    d = int(np.sqrt(2*len(v)))
    def index(i,j): return (max(i,j)*(max(i,j)+1))//2+min(i,j)      
    return ad.array([[v[index(i,j)]/_sym_iso(i,j) for i in range(d)] for j in range(d)])


def eigh(m):
    """np.linalg.eigh, with geometry first"""
    λ,U = np.linalg.eigh(np.moveaxis(m,(0,1),(-2,-1)))
    return np.moveaxis(λ,-1,0), np.moveaxis(U,(-2,-1),(0,1))

def g(μ,X,ret='value_gradient_hessian'):
    """
    Evaluate the function Tr(μ) + | (X-μ)_+ |_Fr^2, 
    as well as its gradient and Hessian. If μ has a smaller 
    size than X, then it is subtracted from the top-left corner.
    - μ : symmetric matrix
    - X : symmetric matrix
    - ret (string, optional) : 'value', 'value_gradient_hessian', 'value_gradient_direction'
    If μ.ndim == X.ndim-1, then μ is assumed to be in flattened form
    """
    μflt = μ.ndim==X.ndim-1
    if μflt: μ = expsym_iso(μ)
    
    # Construct Δ = X-μ
    n=len(μ)
    Δ = X.copy()
    Δ[:n,:n]-=μ 
    
    # Extract the positive part
    λ,U = eigh(Δ)
    λp = np.maximum(0,λ) 
    value = np.einsum('ii...',μ) + np.sum(λp**2)/2 # Value of the functional g

    if ret=='value': return value
        
    Δp = np.einsum('ik...,k...,jk...->ij...',U,λp,U) # (X-μ)_+
    gradient = np.eye(n).reshape((n,n)+(1,)*(X.ndim-2)) - Δp[:n,:n]
    num = λp[None,:]+λp[:,None]
    den = np.abs(λ[None,:])+np.abs(λ[:,None])
    den[den==0]=1
    Λ = num/den
    V = U[:n]
    hessian = np.einsum("ij...,ki...,lj...,mi...,nj...->klmn...",Λ,V,V,V,V)

    if μflt: 
        gradient = fltsym_iso(gradient)
        hessian = fltsym_iso(np.moveaxis(fltsym_iso(hessian),0,2))
        if ret=='value_gradient_direction': return value, gradient, -lp.solve_AV(hessian,gradient)
    assert ret=='value_gradient_hessian'
    return value,gradient,hessian

def prox_perspective_matrix(τ,ρ,m,niter=None):
    """
    Proximal operator of the perspective function.
    - τ (real) : proximal time step
    - ρ (array) : symmetric array of shape (d,d,*s)
    - m (array) : array of shape (d,n,*s)
    - niter (optional, int) : if specified, a fixed number of basic Newton iterations are applied
    (Otherwise, using a damped Newton method with automatic stopping criterion.)
    """
    if τ!=1: 
        ρ1,m1 = prox_perspective_matrix(1,ρ/τ,m/τ,niter)
        return τ*ρ1,τ*m1
    # Build the block matrix
    d,n = m.shape[:2]
    shape = m.shape[2:]
    cat = np.concatenate
    eye = np.broadcast_to(np.eye(n,n).reshape( (n,n)+(1,)*(ρ.ndim-2)), (n,n)+shape)
    s2 = np.sqrt(2)
    X = cat( (cat((eye,np.moveaxis(m/s2,0,1)),axis=1), # (n,n),(n,d)
              cat((m/s2,-ρ),axis=1)), axis=0) # (d,n), (d,d)   

    # Solve the dual problem for μ using a Newton method
    μ = np.zeros(((n*(n+1))//2, *shape))
    if niter is None:
        μ = ad.Optimization.newton_minimize(lambda μ : g(μ,X,'value'), μ, 
        f_value_gradient_direction = lambda μ : g(μ,X,'value_gradient_direction'))
    else:
        for iter in range(niter):
            val,grad,desc = g(μ,X,'value_gradient_direction')
            μ += desc
    
    # Extract the solution
    X[:n,:n]-= expsym_iso(μ) 
    λ,U = eigh(X)
    λm = np.maximum(0,-λ) 
    Δm = np.einsum('ik...,k...,jk...->ij...',U,λm,U) # (X-μ)_-
#    assert np.allclose(μ,Δm[0,0]) # Guaranteed from the optimality conditions
    return Δm[n:,n:], -Δm[n:,:n]*s2


def prox_perspective_matrix_rotated(τ,ρ,m,**kwargs):
    """
    This version is useful if m has size d,n with n>d. 
    (Reduces to size d,d using rotational invariance)
    """
    R,m1 = np.linalg.qr(m.T)
    ρ_,m_ = prox_perspective_matrix(τ,ρ,m1.T,**kwargs)
    return ρ_,m_@R.T

## 1. Validation